# KDD Process Volcano Data Analysis

In [45]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

## Data Cleaning and Preprocessing

### A FAIRE:

Data Collection: Create a notebook, upload the data and store it in a structure that allows
data manipulation. Explore the data using the data summarization methods provided by Python to
fully understand it. You can use the statements we have learned in the course. At this stage, you should
at least show the number of columns and data types in each column, the number of rows and columns
in the dataset, the number of missing data points in each column, and the ranking between variables.
You can use the markdown boxes to describe the dataset at your disposal and some visual components
to help you understand your data. This last result can be used in the next step.


1. Selection
PDF Definition: Creating a target dataset by selecting a subset of variables or data samples on which discovery is to be performed.

Your Project: This is what Person 1 does when they choose to ignore the metadata rows and select specific columns like Year, VEI, Deaths, and Country from the massive TSV file.

2. Preprocessing
PDF Definition: Cleaning the data. This involves removing noise and handling missing data fields.

Your Project: This is the logic in your load_and_clean_data() function:

converting "[]" or empty strings to NaN

filling NaN in the "Deaths" column with 0

ensuring "Year" is a number.

3. Transformation
PDF Definition: Finding useful features to represent the data depending on the goal. This might involve dimensionality reduction or data transformation.

Your Project:

Person 6 (Correlation): Converting Damage_Millions to a Log Scale because the values vary too wildly to plot normally.

Person 4 (Time): Binning the Year data into centuries or decades to make the histogram readable.

4. Data Mining
PDF Definition: Searching for patterns of interest in a particular representational form (e.g., clustering, regression, classification).

Your Project: This is the core logic inside your component functions:

Person 3: Mapping geospatial clusters (Where are the volcanoes?).

Person 6: Finding the correlation (Does high VEI always equal high Damage?).

# 1. Data Collection and Exploration

[cite_start]In this section, we implement the first step of the KDD process: Data Collection and Preprocessing[cite: 15, 46].
[cite_start]The dataset used is the "Significant Volcanic Eruption Database" provided by the National Center for Environmental Information (NCEI)[cite: 22]. [cite_start]It contains over 600 eruptions with attributes such as location, time, VEI (Volcanic Explosivity Index), and impact metrics (deaths, damage)[cite: 23].

**Objectives of this step:**
1.  Load the data into a Pandas DataFrame.
2.  Clean the data by handling missing values and correct data types.
3.  Summarize the data (shape, types, missing values) to understand its quality.
4.  [cite_start]Establish rankings between variables to identify initial patterns.

In [46]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

#Display settings to ensure we see all columns during exploration
pd.set_option('display.max_columns', None)

In [24]:
def load_raw_data(filepath):
    """
    Reads the TSV file.
    Input: filepath (str)
    Output: raw dataframe (pd.DataFrame)
    """
    try:
        df = pd.read_csv(filepath, sep='\t')
        print("File loaded.")
        return df
    except FileNotFoundError:
        print("File not found. Please check the path.")
        return None


file_path = "volcano-events.tsv"
df_raw = load_raw_data(file_path)

df_raw.head(5)

File loaded.


,Search Parameters,Year,Mo,Dy,Tsu,Eq,Name,Location,Country,Latitude,Longitude,Elevation (m),Type,VEI,Agent,Deaths,Death Description,Missing,Missing Description,Injuries,Injuries Description,Damage ($Mil),Damage Description,Houses Destroyed,Houses Destroyed Description,Total Deaths,Total Death Description,Total Missing,Total Missing Description,Total Injuries,Total Injuries Description,Total Damage ($Mil),Total Damage Description,Total Houses Destroyed,Total Houses Destroyed Description
0,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,-4360.0,NaN,NaN,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,-178.475,238.0,Caldera,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,-4350.0,NaN,NaN,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,130.305,704.0,Caldera,7.0,P,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
3,NaN,-4050.0,NaN,NaN,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,-86.165,594.0,Caldera,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,-4000.0,NaN,NaN,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,150.516,724.0,Caldera,6.0,T,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN


In [25]:
def show_structure(df):
    """
    Prints the shape and data types of the dataset.
    Input: df (pd.DataFrame)
    """ 
    rows, cols = df.shape
    print(f"Number of Rows: {rows}")
    print(f"Number of Columns: {cols}")
 
    print("Data Types:")
    print(df.dtypes)

show_structure(df_raw)

Number of Rows: 888
Number of Columns: 35
Data Types:
Search Parameters                      object
Year                                  float64
Mo                                    float64
Dy                                    float64
Tsu                                   float64
Eq                                    float64
Name                                   object
Location                               object
Country                                object
Latitude                              float64
Longitude                             float64
Elevation (m)                         float64
Type                                   object
VEI                                   float64
Agent                                  object
Deaths                                float64
Death Description                     float64
Missing                               float64
Missing Description                   float64
Injuries                              float64
Injuries Description      

In [26]:
def clean_data(df):
    """
    Applies preprocessing: renaming columns, handling types, and filling NaNs.
    Input: df (pd.DataFrame) : The raw dataframe
    Output: df (pd.DataFrame) : The cleaned dataframe
    """
    #Rename columns to remove spaces and special characters for easier coding 
    #and for a better understanding
    df = df.rename(columns={
        'Mo': 'Month',
        'Dy': 'Day',
        'Tsu': 'Tsunami',
        'Eq': 'Earthquake',
        'Damage ($Mil)': 'Damage_Millions',
        'Total Damage ($Mil)': 'Total_Damage_Millions',
        'Elevation (m)': 'Elevation',
        'Total Deaths': 'Total_Deaths',
        'Total Injuries': 'Total_Injuries'
    })

    #If 'Deaths' is empty, we assume 0 for the sake of calculation, rather than dropping the row.
    cols_to_fix = ['VEI', 'Deaths', 'Damage_Millions', 'Injuries']
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    #We drop rows without Lat/Lon/Country because we cannot visualize them on a map.
    df = df.dropna(subset=['Latitude', 'Longitude', 'Country'])

    return df

df_cleaned = clean_data(df_raw.copy())
print("Data cleaned.")
df_cleaned.head(5)

Data cleaned.


,Search Parameters,Year,Month,Day,Tsunami,Earthquake,Name,Location,Country,Latitude,Longitude,Elevation,Type,VEI,Agent,Deaths,Death Description,Missing,Missing Description,Injuries,Injuries Description,Damage_Millions,Damage Description,Houses Destroyed,Houses Destroyed Description,Total_Deaths,Total Death Description,Total Missing,Total Missing Description,Total_Injuries,Total Injuries Description,Total_Damage_Millions,Total Damage Description,Total Houses Destroyed,Total Houses Destroyed Description
1,NaN,-4360.0,NaN,NaN,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,-178.475,238.0,Caldera,6.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,-4350.0,NaN,NaN,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,130.305,704.0,Caldera,7.0,P,0.0,3.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,3.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
3,NaN,-4050.0,NaN,NaN,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,-86.165,594.0,Caldera,6.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,-4000.0,NaN,NaN,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,150.516,724.0,Caldera,6.0,T,0.0,1.0,NaN,NaN,0.0,NaN,0.0,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
5,NaN,-3580.0,NaN,NaN,NaN,NaN,Taal,Luzon-Philippines,Philippines,14.002,120.993,311.0,Stratovolcano,6.0,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Now we want to see if the shape changed after the cleaning.

In [27]:
show_structure(df_cleaned)

Number of Rows: 887
Number of Columns: 35
Data Types:
Search Parameters                      object
Year                                  float64
Month                                 float64
Day                                   float64
Tsunami                               float64
Earthquake                            float64
Name                                   object
Location                               object
Country                                object
Latitude                              float64
Longitude                             float64
Elevation                             float64
Type                                   object
VEI                                   float64
Agent                                  object
Deaths                                float64
Death Description                     float64
Missing                               float64
Missing Description                   float64
Injuries                              float64
Injuries Description      

In [28]:
def show_missing_data(df):
    """
    Calculates and prints the number of missing values per column.
    Input: df (pd.DataFrame)
    """
    # Requirement: Show the number of missing data points in each column 
    missing_values = df.isnull().sum()
    
    # Filter to show only columns that actually have missing data
    missing_only = missing_values[missing_values > 0]
    
    print("Missing Data Points per Column:")
    if not missing_only.empty:
        print(missing_only)
    else:
        print("No missing values found in the cleaned dataset columns.")

show_missing_data(df_cleaned)

Missing Data Points per Column:
Search Parameters                     887
Month                                 131
Day                                   191
Tsunami                               705
Earthquake                            808
Agent                                 363
Death Description                     317
Missing                               875
Missing Description                   873
Injuries Description                  763
Damage Description                    639
Houses Destroyed                      837
Houses Destroyed Description          759
Total_Deaths                          422
Total Death Description               291
Total Missing                         875
Total Missing Description             872
Total_Injuries                        783
Total Injuries Description            753
Total_Damage_Millions                 859
Total Damage Description              618
Total Houses Destroyed                828
Total Houses Destroyed Description    732
dt

In [29]:
def show_rankings(df):
    """
    Shows the ranking (correlation) between numerical variables.
    Input: df (pd.DataFrame)
    """
    #We select only numeric columns for correlation analysis
    numeric_df = df.select_dtypes(include=[np.number])

    corr_matrix = numeric_df.corr()

    #Let's see what correlates most strongly with Deaths
    if 'Deaths' in corr_matrix.columns:
        print("Ranking of variables correlated with 'Deaths':")
        print(corr_matrix['Deaths'].sort_values(ascending=False))

show_rankings(df_cleaned)

Ranking of variables correlated with 'Deaths':
Deaths                                1.000000
Total Missing                         0.861649
Missing                               0.861496
Total Missing Description             0.766464
Missing Description                   0.760213
Total_Deaths                          0.697631
Houses Destroyed                      0.608749
Total Houses Destroyed                0.545423
Injuries Description                  0.457933
Injuries                              0.435376
Total Injuries Description            0.401131
Death Description                     0.383835
Total Death Description               0.353107
Houses Destroyed Description          0.342703
Total Houses Destroyed Description    0.300069
Damage Description                    0.298136
Total_Injuries                        0.284960
Total Damage Description              0.246972
VEI                                   0.151380
Month                                 0.028382
Elevation    

In [ ]:
def investigate_descriptions(df):
    """
    Investigates the content of 'Description' columns to understand why they are numeric
    and how they relate to the absolute counts (like Total_Deaths).
    
    Input: df (pd.DataFrame)
    """
    print("Inspecting Unique Values in Description Columns")
    # We suspect these are discrete classes (1, 2, 3, 4), not random numbers.
    # Let's check the unique values to confirm.
    desc_cols = ['Death Description', 'Damage Description']
    
    for col in desc_cols:
        if col in df.columns:
            unique_vals = sorted(df[col].dropna().unique())
            print(f"Unique values in '{col}': {unique_vals}")
    
    print()

    print("Testing Hypothesis: Is 'Death Description' a proxy for 'Total_Deaths'?")
    
    # We will group the data by 'Death Description' and see the range of 'Total_Deaths'
    if 'Total_Deaths' in df.columns and 'Death Description' in df.columns:
        
        # Group by the description class and calculate stats for the actual death count
        description_stats = df.groupby('Death Description')['Total_Deaths'].agg(
            Count='count', 
            Min_Deaths='min', 
            Median_Deaths='median', 
            Max_Deaths='max',
            Mean_Deaths='mean'
        )
        print(description_stats)
    
    print()
    
    #Correlation Check
    if 'Total_Deaths' in df.columns and 'Death Description' in df.columns:
        corr = df[['Total_Deaths', 'Death Description']].corr().iloc[0,1]
        print(f"Correlation between 'Total_Deaths' and 'Death Description': {corr:.4f}")

investigate_descriptions(df_cleaned)

Inspecting Unique Values in Description Columns
Unique values in 'Death Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Unique values in 'Damage Description': [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
------------------------------
Testing Hypothesis: Is 'Death Description' a proxy for 'Total_Deaths'?
                   Count  Min_Deaths  Median_Deaths  Max_Deaths  Mean_Deaths
Death Description                                                           
1.0                  336         1.0            3.0     15000.0    52.375000
2.0                   26        54.0           68.0       226.0    77.615385
3.0                   43       106.0          215.0      1000.0   321.860465
4.0                   38      1001.0         2000.0     60000.0  7042.894737
------------------------------
Correlation between 'Total_Deaths' and 'Death Description': 0.3785


In [31]:
import plotly.express as px

def visualize_description_vs_deaths(df):
    """
    Creates a box plot to visually compare the 'Death Description' categories
    against the actual 'Total_Deaths' numbers.
    """
    plot_df = df.dropna(subset=['Death Description', 'Total_Deaths']).copy()
    
    # Convert Description to string so it's treated as a category (Class 1, Class 2...), not a continuous number
    plot_df['Category'] = plot_df['Death Description'].astype(str)
    
    fig = px.box(
        plot_df, 
        x='Category', 
        y='Total_Deaths',
        title="Distribution of Deaths by Death Description Category",
        labels={'Category': 'Death Description Class (1-4)', 'Total_Deaths': 'Confirmed Deaths (Log Scale)'},
        category_orders={'Category': ["1.0", "2.0", "3.0", "4.0"]}, # Order the bars logically
        log_y=True, # Use Log scale because volcanic deaths vary wildly (from 1 to 30,000)
        points="all", 
        template='plotly_white'
    )
    
    fig.show()

visualize_description_vs_deaths(df_cleaned)

In [32]:
def analyze_all_description_pairs(df):
    """
    Loops through multiple metric/description pairs to verify if they all follow 
    the ordinal scale pattern (1-4).
    """
    # We want to investigate the pairs as followed :  (Metric Column, Description Column)
    pairs_to_check = [
        ('Missing', 'Missing Description'),
        ('Injuries', 'Injuries Description'),
        ('Houses Destroyed', 'Houses Destroyed Description'),
        ('Total_Damage_Millions', 'Total Damage Description') 
    ]

    for metric_col, desc_col in pairs_to_check:
        
        # Check if columns exist in the dataframe before proceeding
        if metric_col in df.columns and desc_col in df.columns:
            print(f"Analysis: {metric_col}  vs.  {desc_col}")

            
            #Statistical Check
            # Group by the description (1, 2, 3, 4) and see the range of the actual numbers
            stats = df.groupby(desc_col)[metric_col].agg(
                Count='count', 
                Min='min', 
                Median='median', 
                Max='max',
                Mean='mean'
            )
            print(stats)
            
            # Filter data for plotting
            plot_df = df.dropna(subset=[metric_col, desc_col]).copy()
            plot_df['Category'] = plot_df[desc_col].astype(str) # Convert to string for categorical plotting
            
            fig = px.box(
                plot_df, 
                x='Category', 
                y=metric_col,
                title=f"Distribution of {metric_col} by {desc_col}",
                labels={'Category': 'Severity Class (1-4)', metric_col: f'{metric_col} (Log Scale)'},
                category_orders={'Category': ["1.0", "2.0", "3.0", "4.0"]}, 
                log_y=True, # Log scale is crucial because values vary from 1 to Millions
                points="all", 
                template='plotly_white'
            )
            fig.show()
            
        else:
            print(f"Skipping pair {metric_col}/{desc_col}: Columns not found.")

analyze_all_description_pairs(df_cleaned)

Analysis: Missing  vs.  Missing Description
                     Count     Min  Median     Max    Mean
Missing Description                                       
1.0                      5     2.0     9.0    44.0    15.6
2.0                      1    78.0    78.0    78.0    78.0
3.0                      2   120.0   174.5   229.0   174.5
4.0                      2  1500.0  1627.5  1755.0  1627.5


Analysis: Injuries  vs.  Injuries Description
                      Count  Min  Median      Max         Mean
Injuries Description                                          
1.0                      93  0.0     4.0     50.0     9.526882
2.0                      12  0.0    53.0     86.0    37.833333
3.0                      16  0.0   203.0   1000.0   242.375000
4.0                       3  0.0  1972.0  10000.0  3990.666667


Analysis: Houses Destroyed  vs.  Houses Destroyed Description
                              Count     Min  Median     Max         Mean
Houses Destroyed Description                                            
1.0                              19     1.0    10.0    75.0    18.894737
2.0                               5    60.0    63.0    90.0    71.000000
3.0                              14   144.0   300.0   800.0   377.428571
4.0                              12  1109.0  3085.0  9000.0  3593.166667


Analysis: Total_Damage_Millions  vs.  Total Damage Description
                          Count   Min   Median     Max        Mean
Total Damage Description                                          
1.0                           1   1.0    1.000     1.0    1.000000
2.0                           8   2.0    2.750     4.0    2.908000
3.0                           7   5.0   15.000    20.0   14.714286
4.0                          12  67.0  192.684  2000.0  424.364000


Based on the NOAA Significant Volcanic Eruption Database and the classification standards by Simkin and Siebert (1994), the Agent column indicates the specific volcanic hazard or phenomenon that caused fatalities, injuries, or damage during an eruption.
P: Pyroclastic flow or surge (a fast-moving current of hot gas and volcanic matter).

M: Mudflow or Lahar (volcanic mudflow).

T: Tsunami (generated by the eruption).

L: Lava flow.

G: Gas (toxic volcanic gases).

F: Tephra/Ash fall (falling volcanic rock and ash).

A: Avalanche (debris avalanche or landslide).

E: Electrical (lightning associated with the eruption).

I: Indirect causes (such as starvation, disease, or exposure resulting from the eruption).

S: Seismic activity (earthquakes related to the eruption).

### Interpretation of Missing Data & Variable Selection

**1. Observations on Data Quality:**
Our analysis reveals a significant amount of missing data across the dataset. However, a deeper inspection allows us to categorize these gaps into two types:
* Irrelevant/Noise: Columns like Search Parameters (887 missing) are largely empty and do not contain information useful for our visualization goals.
* Redundant Proxies: Columns like Death Description or Damage Description appeared at first to be sparse floats. Our investigation reveals these are actually Ordinal Severity Scales (ranked 1 to 4) rather than continuous measurements.
* We are also going to drop Month and Day because the exact date is not necessary and relevant especially since a part of the dataset doesn't have this data.

**2. Decision on "Description" Columns:**
* We confirmed a high correlation between Death Description and Total_Deaths. For example, a "Description" value of 4 consistently corresponds to catastrophic events with high death tolls.
* We will exclude these description columns from our final dashboard dataset.
* They are redundant. Since we have the precise absolute numbers in Total_Deaths and Total_Damage_Millions, keeping the simplified 1-4 scale would introduce multicollinearity (repetition) without adding precision. We prioritize the exact figures for clearer visualizations ("5,000 deaths" is more informative to a user than "Severity Level 3").

**3. Decision on Flags :**
* Columns like Tsunami and Earthquake are "flags" often left blank when the event did not occur.
We will keep them in the Dataframe and analyze later if they can be useful to our study

**4. Statistical Caution (Means vs. Totals):**
* Because missing values in impact columns (Total_Deaths) are frequent and likely represent "zero" or "unknown" in historical contexts, calculating an average would be misleading.
We will avoid using the mean as a central KPI. Instead, we will use Sums (Total Global Deaths) and Counts (Number of Eruptions) which remain robust even with sparse historical records.

In [33]:
def remove_unnecessary_columns(df):
    """
    Removes columns deemed irrelevant, sparse, or redundant for the visualization dashboard.
    
    Why we are doing this:
    'Search Parameters': Contains metadata about the database query, not the volcano itself.
    'Month' & 'Day': Too granular. We are analyzing historical trends over centuries (by Year), 
    so specific dates are noise for this high-level overview.
    'Description' Columns : We proved these are Ordinal Scales (1-4)
    that are redundant because we already have the precise metrics (e.g., 'Total_Deaths').
    Removing them prevents multicollinearity and cleans the dataset.
       
    Input: df (pd.DataFrame)
    Output: df (pd.DataFrame) - The reduced and final dataframe.
    """
    cols_to_drop = ['Search Parameters', 'Month', 'Day']
    
    #Dynamically find all 'Description' columns
    description_cols = [col for col in df.columns if 'Description' in col]
    
    all_cols_to_remove = cols_to_drop + description_cols
    
    # errors='ignore' ensures the code doesn't crash if we accidentally run it twice (and columns are already gone)
    df = df.drop(columns=all_cols_to_remove, errors='ignore')
    
    print(f"Removed {len(all_cols_to_remove)} columns: {all_cols_to_remove}")
    print(f"Remaining columns: {df.columns.tolist()}")
    
    return df

df_final = remove_unnecessary_columns(df_cleaned)
df_final.head()

Removed 13 columns: ['Search Parameters', 'Month', 'Day', 'Death Description', 'Missing Description', 'Injuries Description', 'Damage Description', 'Houses Destroyed Description', 'Total Death Description', 'Total Missing Description', 'Total Injuries Description', 'Total Damage Description', 'Total Houses Destroyed Description']
Remaining columns: ['Year', 'Tsunami', 'Earthquake', 'Name', 'Location', 'Country', 'Latitude', 'Longitude', 'Elevation', 'Type', 'VEI', 'Agent', 'Deaths', 'Missing', 'Injuries', 'Damage_Millions', 'Houses Destroyed', 'Total_Deaths', 'Total Missing', 'Total_Injuries', 'Total_Damage_Millions', 'Total Houses Destroyed']


,Year,Tsunami,Earthquake,Name,Location,Country,Latitude,Longitude,Elevation,Type,VEI,Agent,Deaths,Missing,Injuries,Damage_Millions,Houses Destroyed,Total_Deaths,Total Missing,Total_Injuries,Total_Damage_Millions,Total Houses Destroyed
1,-4360.0,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,-178.475,238.0,Caldera,6.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,-4350.0,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,130.305,704.0,Caldera,7.0,P,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,-4050.0,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,-86.165,594.0,Caldera,6.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,-4000.0,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,150.516,724.0,Caldera,6.0,T,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
5,-3580.0,NaN,NaN,Taal,Luzon-Philippines,Philippines,14.002,120.993,311.0,Stratovolcano,6.0,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## Spatial Analysis

In [ ]:
# Function: Build a scatter-based world map showing individual volcano eruptions.
# Input:
#   - df (pd.DataFrame): filtered volcano dataset containing at least
#       ["Latitude", "Longitude", "Type", "Name", "VEI", "Country", "Year", "Deaths"]
# Output:
#   - fig (plotly.graph_objs.Figure): an interactive scatter_geo map (used in Dash)

def get_map_points(df):
    # Copy the dataframe to avoid modifying the original one
    df_map = df.copy()

    # Replace missing VEI values with a small default value (0.5)
    # to avoid invisible points on the map
    df_map['VEI_Size'] = df_map['VEI'].fillna(0.5)

    # Create the base geographic scatter plot
    fig = px.scatter_geo(
        df_map,
        lat="Latitude",           # latitude of volcano
        lon="Longitude",          # longitude of volcano
        color="Type",             # volcano morphological category
        size="VEI_Size",          # bubble size based on VEI (explosivity)
        hover_name="Name",        # volcano name in tooltip
        hover_data={              # additional tooltip information
            "Country": True,
            "Year": True,
            "Deaths": True,
            "VEI": True,
            "VEI_Size": False
        },
        title="Global Volcano Distribution (Bubble size = VEI)",
        projection="natural earth", # projection style
        size_max=15,                # maximum bubble size
        template="plotly_dark"      # dark theme to match dashboard
    )

    # Custom color palette: 20 vivid volcanic colors (orange → red → magenta → violet)
    warm_palette = [
   
    "#ffd500",  # deep orange
    "#ff8f00",  # vivid orange
    "#ff3d00",  # bright red-orange
    "#ff1a00",  # pure red-orange
    "#e60000",  # intense red
    "#c51162",  # magenta
    "#ff0055",  # neon pink-red
    "#d81b60",  # pink-magenta
    "#b0003a",  # dark magenta-red
    "#9c004d",  # deep pink-purple
    "#aa00ff",  # neon violet
    "#8e24aa",  # classic violet
    "#7b1fa2",  # deep violet
    "#6a1b9a",  # darker violet
    "#4a148c",  # almost purple-black
    "#7f0000",
    "#b30000",
    "#d50000",
    "#ff1744",
    "#ff4081"
    ]

    n_traces = len(fig.data)   # one trace per volcano Type
    n_colors = len(warm_palette)

    # Assign a distinct color from the palette to each volcano Type
    for i, trace in enumerate(fig.data):
        color = warm_palette[i % n_colors]     # loop through palette if Types > 20
        trace.marker.update(
            color=color,
            line=dict(width=0)                 # remove outline for cleaner look
        )

    # Add geographic features (coastlines, countries, land)
    fig.update_geos(
        showcountries=True,
        showcoastlines=True,
        showland=True,
    )

    # Transparent background to match the dashboard's dark theme
    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )

    return fig

In [48]:
# Function: Build a choropleth (colored world map) aggregated by country.
# Input:
#   - df (pd.DataFrame): filtered volcano dataset containing at least:
#       ["Country", value_col]
#   - value_col (str): column name used for coloring (e.g., "Count", "Deaths", "Damage_Millions")
#   - title (str): title of the choropleth
#   - color_label (str): label for the colorbar (e.g., "Eruptions", "Deaths", "Damage")
# Output:
#   - fig (plotly.graph_objs.Figure): an interactive choropleth map (used in Dash)

def get_country_choropleth(df, value_col, title, color_label):
    # Aggregate data by country for the selected metric (count, deaths, damage…)
    agg = df.groupby('Country', as_index=False)[value_col].sum()
    # Custom continuous volcanic colormap (dark red → orange → magenta → violet)
    # Designed to avoid white/yellow and emphasize bright volcanic tones
    volcano_scale = [
        (0.00, "#000000"),   # very dark base
        (0.05, "#4b0000"),   # deep red
        (0.10, "#7f0000"),   # darker red
        (0.20, "#b00000"),   # intense red
        (0.30, "#d50000"),   # bright red
        (0.40, "#ff1400"),   # red-orange (flashy)
        (0.55, "#ff3d00"),   # bright orange-red
        (0.70, "#ff6d00"),   # orange incandescent
        (0.85, "#ff8500"),   # bright orange
        (0.93, "#d81b60"),   # magenta
        (1.00, "#6a1b9a")    # deep violet
    ]
    # Build the choropleth map
    fig = px.choropleth(
        agg,
        locations='Country',            # country name column
        locationmode='country names',   # match names to world countries
        color=value_col,                # metric used for color intensity
        hover_name='Country',           # tooltip title
        title=title,
        labels={value_col: color_label}, # name of the color axis
        template="plotly_dark",          # dark theme
        color_continuous_scale=volcano_scale,
        projection="natural earth"       # projection style
    )
    # Display country borders, coastlines, and land
    fig.update_geos(
        showcountries=True,
        showcoastlines=True,
        showland=True
    )
    # Transparent background to blend with the dashboard
    fig.update_layout(
        margin={"r": 0, "t": 50, "l": 0, "b": 0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )

    return fig


## Temporal Analysis

In [49]:
def get_frequency_figure(df):
    fig = px.histogram(
        df, 
        x="Year", 
        title="Eruption Frequency",
        nbins=100,
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722')
    fig.update_layout(
        xaxis_title="Year", 
        yaxis_title="Count",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Impact Analysis

In [50]:
def get_impact_figure(df):
    top_deadly = df.nlargest(10, 'Deaths').sort_values('Deaths', ascending=True)
    fig = px.bar(
        top_deadly,
        x="Deaths",
        y="Name",
        orientation='h',
        text="Deaths",
        title="Top 10 Deadliest Eruptions",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722', textposition='outside')
    fig.update_layout(
        xaxis_title="Deaths", 
        yaxis_title="",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Correlation Analysis

In [51]:
def get_correlation_figure(df):
    damage_df = df[df['Damage_Millions'] > 0].copy()
    if damage_df.empty:
         return px.scatter(title="No Data")

    fig = px.scatter(
        damage_df,
        x="VEI",
        y="Damage_Millions",
        size="Deaths",
        hover_name="Name",
        log_y=True,
        title="VEI vs. Impact",
        template="plotly_dark"
    )
    fig.update_traces(marker=dict(color='#ff5722', opacity=0.7))
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## UI for Dashboard

A Visualization Dashboard: Use Python Dash (https://dash.plotly.com/) to visualize a
dashboard containing the four elements built in the previous step. The dashboard should include the
names of the team members and the name of the dataset used for this project and the description of
the project objective. Remember that a goal should be concise, achievable, and tangible.

In [52]:
# Initialize App
app = Dash(__name__)

# Load Data
df = load_data()

# Styles
SIDEBAR_STYLE = {
    "position": "fixed",
    "top": 0,
    "left": 0,
    "bottom": 0,
    "width": "16rem",
    "padding": "2rem 1rem",
    "background-color": "#111111",
    "color": "white"
}

CONTENT_STYLE = {
    "margin-left": "18rem",
    "margin-right": "2rem",
    "padding": "2rem 1rem",
    "background-color": "#000000",
    "min-height": "100vh",
    "color": "white"
}

CARD_STYLE = {
    "background-color": "#1e1e1e",
    "padding": "20px",
    "border-radius": "10px",
    "margin-bottom": "20px",
    "box-shadow": "0 4px 6px rgba(0,0,0,0.3)"
}

# Layout
app.layout = html.Div([
    # Sidebar
    html.Div([
        html.H2("Volcano Insights", style={'font-size': '20px', 'margin-bottom': '20px', 'color': '#ff5722'}),
        html.Hr(style={'border-color': '#333'}),
        html.P("Filters", style={'color': '#888'}),
        
        html.Label("Year Range", style={'margin-top': '20px'}),
        dcc.RangeSlider(
            id='year-slider',
            min=df['Year'].min(),
            max=df['Year'].max(),
            value=[df['Year'].min(), df['Year'].max()],
            marks={str(year): str(year) for year in range(int(df['Year'].min()), int(df['Year'].max()), 1000)},
            tooltip={"placement": "bottom", "always_visible": True},
            className="dark-slider"
            
        ),
        
        html.Label("Country", style={'margin-top': '20px'}),
        dcc.Dropdown(
            id='country-dropdown',
            options=[{'label': c, 'value': c} for c in sorted(df['Country'].unique())],
            placeholder="All Countries",
            style={'color': 'black'} # Dropdown text needs to be black to be visible on white bg of default dropdown
        )
    ], style=SIDEBAR_STYLE),

    # Main Content
    html.Div([
        html.H1("Volcano Insights Dashboard", style={'margin-bottom': '5px'}),
        html.P("Analyzing Significant Volcanic Eruptions", style={'color': '#888', 'margin-bottom': '30px'}),

        # KPI Row
        html.Div([
            html.Div([
                html.H4("Total Eruptions", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-eruptions', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Deaths", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-deaths', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Damage ($M)", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-damage', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}),

        # Charts Row 1
       html.Div([
    # === BIG MAP CARD ===
    html.Div(
        [
            # Choix du type de carte
            html.Div([
                html.Label("Map view:", style={'color': 'white', 'margin-right': '10px'}),
                dcc.RadioItems(
                    id='map-mode',
                    options=[
                        {'label': 'Eruptions (points)', 'value': 'points'},
                        {'label': 'Eruptions / country', 'value': 'eruptions_country'},
                        {'label': 'Deaths / country', 'value': 'deaths_country'},
                        {'label': 'Damage / country', 'value': 'damage_country'},
                    ],
                    value='points',
                    inline=True,
                    className='dark-radio'
                )
            ], style={'margin-bottom': '10px'}),

            # La figure 
            dcc.Graph(id='map-graph', style={'height': '650px', 'width': '100%'})
        ],
        style={**CARD_STYLE, 'flex': '2', 'margin-right': '20px', 'padding': '10px'}
    ),

    # === CARD DE DROITE (time graph comme avant) ===
    html.Div(
        [dcc.Graph(id='time-graph', style={'height': '650px', 'width': '100%'})],
        style={**CARD_STYLE, 'flex': '1', 'padding': '10px'}
    )
], style={'display': 'flex', 'margin-bottom': '20px'}),

        # Charts Row 2
        html.Div([
            html.Div([dcc.Graph(id='impact-graph')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='corr-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex'})

    ], style=CONTENT_STYLE)
])

# Callbacks
@app.callback(
    [Output('map-graph', 'figure'),
     Output('time-graph', 'figure'),
     Output('impact-graph', 'figure'),
     Output('corr-graph', 'figure'),
     Output('kpi-eruptions', 'children'),
     Output('kpi-deaths', 'children'),
     Output('kpi-damage', 'children')],
    [Input('country-dropdown', 'value'),
     Input('year-slider', 'value'),
     Input('map-mode', 'value')] 
     
)
def update_dashboard(selected_country, year_range, map_mode):
    dff = df.copy()

    # Filtre pays
    if selected_country:
        dff = dff[dff['Country'] == selected_country]

    # Filtre années
    if year_range:
        dff = dff[(dff['Year'] >= year_range[0]) & (dff['Year'] <= year_range[1])]

    # KPIs
    total_eruptions = len(dff)
    total_deaths = f"{int(dff['Deaths'].sum()):,}"
    total_damage = f"${dff['Damage_Millions'].sum():,.0f}"

    # === CHOIX DE LA CARTE SELON map_mode ===
    if map_mode == 'points':
        fig_map = get_map_points(dff)

    elif map_mode == 'eruptions_country':
        dff_counts = dff.copy()
        dff_counts['Count'] = 1
        fig_map = get_country_choropleth(
            dff_counts, 'Count',
            title="Number of eruptions per country",
            color_label="Eruptions"
        )

    elif map_mode == 'deaths_country':
        fig_map = get_country_choropleth(
            dff, 'Deaths',
            title="Total deaths per country",
            color_label="Deaths"
        )

    elif map_mode == 'damage_country':
        fig_map = get_country_choropleth(
            dff, 'Damage_Millions',
            title="Total damage per country (Million USD)",
            color_label="Damage (M$)"
        )

    else:
        fig_map = get_map_points(dff)

    # Les autres figures comme avant
    fig_time = get_frequency_figure(dff)
    fig_impact = get_impact_figure(dff)
    fig_corr = get_correlation_figure(dff)

    return fig_map, fig_time, fig_impact, fig_corr, total_eruptions, total_deaths, total_damage

if __name__ == '__main__':
    print("Launching Dashboard...")
    print("Dashboard launched at: http://127.0.0.1:7860")
    app.run(host='127.0.0.1', port=7860, debug=True)

NameError: name 'load_data' is not defined